# Week 1: Delivery-time regression

## Objective
Predict the cleaned dataset's `time_taken_min` target from information that could plausibly be available at order placement or assignment. Compare a Linear Regression baseline with a Random Forest Regressor using a chronological holdout.

### Target and feature policy
Target: `time_taken_min` (the source column `Time_taken (min)` after preprocessing). Predictors include approximate delivery distance, traffic, weather, vehicle type and condition, multiple-delivery count, city, order hour, day of week, festival, order day, and driver age where present. Excluded: order and driver IDs, target, driver ratings, raw timestamps, pickup time/hour, and order-to-pickup duration to avoid identifiers, post-outcome information, and target leakage.

Distance is straight-line great-circle approximation, not road-network distance. The dataset has no promised delivery times, road-network data, fleet availability/capacity, costs, or route event traces. This is an educational retrospective experiment, not a causal claim or production-ready service.

In [1]:
import sys
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/'src/modeling.py').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from modeling import run_modeling
results=run_modeling(ROOT/'data/cleaned_logistics_data.csv',make_figures=True)
print('Implementation:',results['implementation'])
print('Date cutoff:',results['cutoff'].date())
print('Train/test rows:',len(results['train']), '/',len(results['test']))
print('Feature count:',len(results['feature_names']))
display(results['comparison'].round(3))
print('Better model by MAE:',results['better_model'])
display(results['feature_importance'].head(15).rename('importance').to_frame().round(4))
print('Figures saved to visualizations/')

Implementation: NumPy fallback (scikit-learn unavailable)
Date cutoff: 2022-03-29
Train/test rows: 36138 / 9446
Feature count: 36
                     MAE   RMSE     R2
Linear Regression  5.171  6.501  0.531
Random Forest      4.334  5.481  0.666
Better model by MAE: Random Forest
                             importance
road_traffic_density_Low         0.1471
delivery_person_age              0.1425
multiple_deliveries              0.1350
vehicle_condition                0.0872
approx_delivery_distance_km      0.0834
weather_conditions_Sunny         0.0830
road_traffic_density_Jam         0.0814
order_hour                       0.0528
festival_Yes                     0.0306
weather_conditions_Fog           0.0289
weather_conditions_Cloudy        0.0258
festival_No                      0.0187
type_of_vehicle_motorcycle       0.0119
road_traffic_density_Medium      0.0100
city_Metropolitian               0.0094
Figures saved to visualizations/


## Method and evaluation

### Preprocessing
The script retains rows with a target and valid order date. It uses earlier dates for training and later dates for testing, with no date overlap. This better reflects an operational use case: future orders are predicted from past observations. A random split could place same-period patterns in both sets and make generalization look easier than it is. Numeric gaps are filled with training-set medians. Categoricals are filled as `Unknown` and one-hot encoded; test columns are aligned to training categories so unseen values do not leak information.

### Models and metrics
Linear Regression is a simple baseline; Random Forest can represent nonlinear relationships and interactions. MAE is typical absolute error in minutes, RMSE penalizes large misses more strongly, and R² compares residual error with a constant-mean baseline. Lower MAE/RMSE and higher R² are better. Feature importance is predictive split-importance, not causal impact.

## Results and logistics interpretation

![Actual vs predicted](../visualizations/regression_actual_vs_predicted.png)

The diagonal indicates perfect predictions. The chart uses a fixed-seed test sample for readability; metrics use the complete test set.

![Metric comparison](../visualizations/regression_model_performance.png)

![Random Forest feature importance](../visualizations/regression_random_forest_feature_importance.png)

More important model features identify patterns the fitted forest used to reduce prediction error. They do not show that changing a feature will change delivery time. Use these results as a screening baseline for operational data collection and future evaluation, not as dispatch policy.

## Limitations and next data needs

The dataset is historical and covers a limited 2022 period; its labels and coordinate quality limit generalization. Straight-line distance differs from road distance. No promised delivery time means customer on-time performance cannot be evaluated. No road network, route-level traffic, fleet availability, courier capacity, delivery cost, or complete event timestamps are available. A stronger future evaluation should add these fields, use a later untouched time period, and compare model predictions against an operational baseline before any deployment. This notebook does not implement clustering or optimization.